# 07. Is the CIFAR-100 head useful after deletion request 16?

Before trying a seventeenth deletion, check whether the model can still recognize the classes we want to keep, whether the first 16 targets stayed suppressed, and whether it can still recognize the next target. Run 16 deletions, then stop. Measure two fusion strengths on the same updates and development images.

**Decision before the run.** A strength qualifies for the release experiment if all three seeds first exhaust their space at request 16 and pass every configured accuracy limit. Read the per-class results alongside this decision. If both qualify, use the weaker subtraction in the next experiment. If neither qualifies, inspect which criterion failed. A target already weak before its request provides little evidence about the effect of that deletion.

**Resources and scope.** Colab CPU with standard RAM. The frozen backbone processes 50,000 training images once. There are 48 small adapter fits. The float32 feature values occupy about 13 MB before labels and intermediate copies. Allow additional disk space for the CIFAR download, extracted images and model. Runtime and peak disk usage have not been measured. Small results go to R2. This is a development check with one class order and one pretrained backbone.

## Words used in this notebook

- **Class ID and request number.** Class IDs start at zero. Requests 1 through 16 target classes 0 through 15. Class 16 is next.
- **Backbone and head.** The frozen backbone converts each image into 64 numbers. The head uses those numbers to choose among 100 classes.
- **Adapter and rank.** Each adapter learns a small change to the head. Rank 4 lets it use up to four independent input directions.
- **Projector and exhaustion.** The projector restricts each new update to available directions. Zero free directions means exhaustion.
- **Lambda or fusion strength.** This number scales the update when it is subtracted from the original head. Both strengths use identical learned updates.
- **Development split.** Examples reserved before adapter training. The public backbone was pretrained on CIFAR-100, so this separation applies to the adapters. The official test set stays available for a later audit.
- **Fixed retained group.** Classes 30 through 99 are outside the planned 30 requests. We score the same images at every step.
- **Reversal.** The increase in an old target's accuracy from immediately after its own deletion to request 16. A decrease counts as zero reversal.
- **Percentage points.** A rise from 5% to 12% is seven percentage points. Each target has 100 development images.
- **Seed and standard deviation.** A seed sets adapter randomness. The standard deviation measures variation across the three seeds, which share one split.
- **R2 and SHA-256.** R2 stores the results across sessions. A SHA-256 hash checks the contents of a file.
- **Run identity.** The settings, data, code and environment that must agree when continuing an interrupted run.

## 0. Setup on Colab

Clone and install the project from the named main branch. Replace REVISION with a commit
hash to pin it. Setup refuses to proceed after the project has been imported: restart the
runtime first. Existing local checkouts are used as they are. Pip installs missing or
incompatible dependencies; already satisfied dependencies are retained.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "main"   # Replace with a commit hash to pin this run.

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    target = "origin/main" if REVISION == "main" else REVISION
    subprocess.run(["git", "-C", "DSR", "checkout", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

print("working directory:", Path.cwd())

## 1. Load code

Print the code revision and software versions. The package handles splitting, training, measuring, and recovering saved work.

In [ ]:
import json
import tempfile
import matplotlib.pyplot as plt
import pandas as pd
import torch

import src.common as common
from src.common import file_hash, load_config, provenance
from src.run_m0 import baseline_caches
from src.run_m5 import resume_status, write_once
from src.runstore import connect
from src.stage_b_start import development_split, run_start_check, summarize

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()
print(json.dumps(PROV, indent=2))

## 2. Settings

The limits in config.yaml are proposed research cutoffs for this pilot. They require a small loss on retained images, low accuracy on every earlier target, limited reversal, and enough room to improve on the next target. They apply separately to each seed. Changing them after viewing results defines a different experiment.

In [ ]:
CFG = load_config(REPO / "config.yaml")
OPT = CFG["stage_b_start"]
RUN, TASKS, SEEDS = OPT["run_label"], OPT["tasks"], CFG["seeds"]
assert len(SEEDS) >= 3 and CFG["runtime"]["device"] == "cpu"
torch.set_num_threads(CFG["runtime"]["threads"])
print(json.dumps(CFG, indent=2))
print("Deletion classes:", list(range(TASKS)), "| next class:", TASKS)
print("Fixed retained classes:", OPT["fixed_retain_start"], "through", CFG["model"]["classes"] - 1)
print("Seeds:", SEEDS, "| adapter fits:", len(SEEDS)*TASKS, "| backbone:", CFG["model"]["entrypoint"])
print("CPU only. Runtime unmeasured. Training features about 13 MB plus dataset and working space.")

## 3. Connect to storage

Enable the three Colab secrets DSR_ENDPOINT, DSR_ACCESS_KEY_ID and DSR_SECRET_ACCESS_KEY. Retrieve existing results for this label before training. The final verification downloads a fresh copy.

In [ ]:
OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
STORE = connect(required=True)
if RUN in STORE.runs():
    STORE.get(RUN, OUT)
print("Storage:", STORE.bucket, RUN)

## 4. Prepare the fixed development split

Use the official training split. Reserve 100 of the 500 images in every class for development and leave 400 for possible adapter fitting. Only the first 16 classes supply training examples. The deterministic split is shared by all six seed-strength combinations. Fewer fitting examples than M5 means these heads must be rebuilt. The cache manifest and split hash identify the exact inputs.

In [ ]:
folder, cache_id, head, caches = baseline_caches(CFG, PROV, splits=("train",))
fit, dev, split = development_split(caches["train"], OPT, CFG["model"]["classes"])
IDENTITY = {"protocol": "stage-b-request16-development-v1", "configuration": CFG,
            "environment": PROV, "cache_identity": cache_id, "split": split,
            "train_manifest": json.loads((folder / "train.json").read_text()),
            "notebook_sha256": file_hash(REPO / "notebooks/07_cifar100_request16.ipynb")}
print("Checkpoint:", cache_id["checkpoint_sha256"])
print("Split:", split)

## 5. Resume

Skip complete seeds with matching identities and verified records. Each completed task saves an immutable JSONL record. An interrupted seed replays from the beginning and checks earlier records. This rebuilds its weights, random states and projector from the recorded seed and inputs.

In [ ]:
status = resume_status(OUT, IDENTITY, SEEDS, TASKS)
print("Completed:", status["complete"], "| remaining:", status["remaining"])

## 6. Train and measure

After each request, subtract the accumulated updates at both strengths and score the same development images. Record the number correct and the number evaluated for every class, including the clean head at step zero.

In [ ]:
records = run_start_check(CFG, head, fit, dev, IDENTITY, OUT, sync=lambda path: STORE.put(path, RUN))
history, targets, scores = summarize(records, OPT)
assert len(records) == len(SEEDS) * (TASKS + 1)
assert len(history) == len(records) * len(OPT["lambdas"]) * CFG["model"]["classes"]
assert len(targets) == len(SEEDS) * len(OPT["lambdas"]) * TASKS
print("Verified step records:", len(records))

## 7. Interpret the request-16 head

Check the measured free rank first. Then read each target across its row from before deletion to immediately after, then to the boundary at request 16. Inspect negative drops or a low starting accuracy before attributing a change to the request. The fixed-group loss and worst old class determine whether a good average hides a problem.

In [ ]:
print(scores[["seed", "lambda", "free_rank", "exhausted_at_boundary"]].to_string(index=False))
print(targets.round(2).to_string(index=False))
print(scores.round(2).to_string(index=False))
columns = ["retain_clean", "retain_final", "retain_loss_pp", "worst_after", "worst_final",
           "worst_reversal_pp", "next_at_boundary"]
summary = scores.groupby("lambda")[columns].agg(["mean", "std"])
print(summary.round(2).to_string())
eligible = scores.groupby("lambda").passes.all()
choices = eligible.index[eligible].tolist()
decision = {"eligible_strengths": choices, "selected_strength": min(choices) if choices else None}
fig, ax = plt.subplots(figsize=(9, 4))
for strength, part in targets.groupby("lambda"):
    by_class = part.groupby("class_id").at_boundary.agg(["mean", "std"])
    ax.errorbar(by_class.index, by_class["mean"], yerr=by_class["std"], marker="o", label=f"lambda {strength}")
ax.axhline(OPT["max_old_accuracy_pct"], color="black", linestyle=":", label="old-class limit")
ax.set(xlabel="Earlier target (class ID)", ylabel="Accuracy after request 16 (%)",
       title="Earlier targets on development images, mean and standard deviation")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Save and verify

Save raw counts, the per-target comparison, the checks for each seed and the decision. Reopen the tables and verify their sizes. Compare file hashes after downloading the complete run into a temporary directory. Feature tensors stay outside the result folder.

In [ ]:
tables = {"class_history.csv": history, "earlier_targets.csv": targets, "seed_checks.csv": scores}
for name, table in tables.items():
    table.to_csv(OUT / name, index=False)
    restored = pd.read_csv(OUT / name)
    pd.testing.assert_frame_equal(restored, table, check_dtype=False)
summary.to_csv(OUT / "summary.csv")
pd.testing.assert_frame_equal(pd.read_csv(OUT / "summary.csv", header=[0, 1], index_col=0), summary,
                              check_dtype=False)
fig.savefig(OUT / "earlier_targets.png", dpi=120)
write_once(OUT / "metadata.json", {"identity": IDENTITY, "notebook": "07_cifar100_request16.ipynb",
           "decision": decision, "limits": OPT, "scope": "development only, fixed class order and backbone",
           "decision_rule": "all criteria in every seed; choose the smaller qualifying strength",
           "resume": "completed seed; replay interrupted seeds"})
STORE.put(OUT, RUN)
with tempfile.TemporaryDirectory() as tmp:
    copy = STORE.get(RUN, Path(tmp))
    def hashes(root):
        return {p.relative_to(root).as_posix(): file_hash(p) for p in root.rglob("*") if p.is_file()}
    assert hashes(copy) == hashes(OUT)
print("Verified R2 copy:", RUN)
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")

## What the measured results mean

Read the number of passing seeds and the failed checks for each strength. A missing qualifying strength is a reason to examine the starting model. A qualifying strength is a candidate for a separate release experiment. These measurements concern predictions on development images.

In [ ]:
for strength, part in scores.groupby("lambda"):
    passed = int(part.passes.sum())
    print(f"Lambda {strength}: {passed} of {len(part)} seeds met all limits.")
    for key in ["exhausted_at_boundary", "retain_ok", "old_after_ok", "old_final_ok", "reversal_ok", "next_ok"]:
        failed = part.loc[~part[key], "seed"].tolist()
        if failed:
            print("  Failed", key, "for seeds", failed)
if choices:
    print("Use lambda", decision["selected_strength"], "for the planned request-17 comparison.")
else:
    print("Inspect the failed checks before testing release at request 17.")